# Notebook 3a (versión modificada) — CNN para clasificación de imágenes
Dataset: **Fashion-MNIST**. Cambios principales: arquitectura con 3 bloques convolucionales, BatchNormalization, Dropout, aumento de datos, EarlyStopping, comparación contra un MLP, mapas de características de varias prendas y análisis más detallado de errores.

## 0. Configuración inicial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

sns.set_style("white")
SEMILLA = 7
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)
print("TensorFlow:", tf.__version__)

## 1. Cargar y explorar los datos

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

NOMBRES_CLASES = ["Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo",
                  "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"]

print("X_train:", X_train.shape, " X_test:", X_test.shape)

# Una muestra por cada clase
fig, ejes = plt.subplots(2, 5, figsize=(12, 5.5))
for clase, ax in enumerate(ejes.flat):
    idx = np.where(y_train == clase)[0][0]
    ax.imshow(X_train[idx], cmap="magma")
    ax.set_title(NOMBRES_CLASES[clase], fontsize=10)
    ax.axis("off")
plt.suptitle("Una muestra por clase de Fashion-MNIST")
plt.tight_layout()
plt.show()

# Balance de clases
plt.figure(figsize=(9, 3.5))
sns.countplot(x=[NOMBRES_CLASES[c] for c in y_train], color="#2E86AB")
plt.xticks(rotation=45)
plt.title("Distribución de clases en entrenamiento")
plt.ylabel("Cantidad")
plt.tight_layout()
plt.show()

## 2. Preparar los datos

In [ ]:
X_train_norm = (X_train.astype("float32") / 255.0).reshape(-1, 28, 28, 1)
X_test_norm = (X_test.astype("float32") / 255.0).reshape(-1, 28, 28, 1)
print("Forma final:", X_train_norm.shape, "| rango:", X_train_norm.min(), "a", X_train_norm.max())

## 3. Línea base: un MLP (para comparar contra la CNN)
Se aplana la imagen y se usan solo capas densas, ignorando la estructura espacial.

In [ ]:
modelo_mlp = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
], name="mlp_base")
modelo_mlp.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

hist_mlp = modelo_mlp.fit(X_train_norm, y_train, epochs=10, batch_size=128,
                          validation_split=0.2, verbose=0)
prec_mlp = modelo_mlp.evaluate(X_test_norm, y_test, verbose=0)[1]
print(f"MLP -> parámetros: {modelo_mlp.count_params():,} | precisión en test: {prec_mlp:.4f}")

## 4. Arquitectura CNN mejorada
Tres bloques `Conv2D + BatchNorm + MaxPooling + Dropout`, y capas de aumento de datos (solo se activan al entrenar).

In [ ]:
keras.backend.clear_session()
tf.random.set_seed(SEMILLA)

modelo_cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    # Aumento de datos
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    # Bloque 1
    layers.Conv2D(32, 3, padding="same", activation="relu", name="conv_1"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    # Bloque 2
    layers.Conv2D(64, 3, padding="same", activation="relu", name="conv_2"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    # Bloque 3
    layers.Conv2D(128, 3, padding="same", activation="relu", name="conv_3"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    # Clasificador
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax"),
], name="cnn_mejorada")

modelo_cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo_cnn.summary()

## 5. Entrenamiento con EarlyStopping y ReduceLROnPlateau

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]
historial = modelo_cnn.fit(X_train_norm, y_train, epochs=30, batch_size=128,
                           validation_split=0.2, callbacks=callbacks, verbose=1)
print("Épocas entrenadas:", len(historial.history["loss"]))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
for ax, met, tit in zip(ejes, ["loss", "accuracy"], ["Pérdida", "Precisión"]):
    ax.plot(historial.history[met], label="CNN entrenamiento")
    ax.plot(historial.history["val_" + met], label="CNN validación")
    ax.plot(hist_mlp.history["val_" + met], "--", label="MLP validación", color="gray")
    ax.set_title(tit); ax.set_xlabel("Época"); ax.legend()
plt.tight_layout()
plt.show()

perdida_test, prec_cnn = modelo_cnn.evaluate(X_test_norm, y_test, verbose=0)
print(f"CNN -> parámetros: {modelo_cnn.count_params():,} | precisión en test: {prec_cnn:.4f}")
print(f"MLP -> parámetros: {modelo_mlp.count_params():,} | precisión en test: {prec_mlp:.4f}")

## 6. Filtros aprendidos por la primera capa convolucional

In [ ]:
pesos_conv1, _ = modelo_cnn.get_layer("conv_1").get_weights()
print("Forma de los filtros:", pesos_conv1.shape)

fig, ejes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(ejes.flat):
    ax.imshow(pesos_conv1[:, :, 0, i], cmap="coolwarm")
    ax.axis("off")
plt.suptitle("Los 32 filtros 3x3 de conv_1 (azul = peso negativo, rojo = peso positivo)")
plt.tight_layout()
plt.show()

## 7. Mapas de características de varias prendas

In [ ]:
modelo_extractor = keras.Model(
    inputs=modelo_cnn.inputs,
    outputs=[modelo_cnn.get_layer(n).output for n in ["conv_1", "conv_2", "conv_3"]],
)

clases_ejemplo = [1, 5, 9]  # Pantalón, Sandalia, Botín
indices_ejemplo = [int(np.where(y_test == c)[0][0]) for c in clases_ejemplo]
mapas = [modelo_extractor.predict(X_test_norm[i:i + 1], verbose=0) for i in indices_ejemplo]

def mostrar_mapas(pos_capa, nombre_capa, n_mapas=8):
    fig, ejes = plt.subplots(len(indices_ejemplo), n_mapas + 1, figsize=(15, 5))
    for fila, (idx, m) in enumerate(zip(indices_ejemplo, mapas)):
        ejes[fila, 0].imshow(X_test[idx], cmap="gray")
        ejes[fila, 0].set_title(NOMBRES_CLASES[y_test[idx]], fontsize=9)
        ejes[fila, 0].axis("off")
        for j in range(n_mapas):
            ejes[fila, j + 1].imshow(m[pos_capa][0, :, :, j], cmap="inferno")
            ejes[fila, j + 1].axis("off")
    plt.suptitle(f"Mapas de características — {nombre_capa} (primeros {n_mapas} filtros)")
    plt.tight_layout()
    plt.show()

mostrar_mapas(0, "conv_1")
mostrar_mapas(2, "conv_3")

## 8. Evaluación: matriz de confusión y métricas por clase

In [ ]:
y_pred_probs = modelo_cnn.predict(X_test_norm, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=NOMBRES_CLASES, normalize="true",
    ax=ax, cmap="Purples", xticks_rotation=45, colorbar=False, values_format=".2f")
plt.title("Matriz de confusión normalizada — CNN mejorada")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=NOMBRES_CLASES, digits=3))

In [ ]:
# Precisión por clase
prec_clase = [(y_pred[y_test == c] == c).mean() for c in range(10)]
orden = np.argsort(prec_clase)
plt.figure(figsize=(9, 4))
plt.barh([NOMBRES_CLASES[i] for i in orden], [prec_clase[i] for i in orden], color="#E07B39")
plt.xlim(0.6, 1.0)
plt.xlabel("Precisión por clase")
plt.title("¿Qué prendas le cuestan más a la CNN?")
plt.tight_layout()
plt.show()

## 9. Errores más "seguros": donde la red se equivocó con mayor confianza

In [ ]:
indices_error = np.where(y_pred != y_test)[0]
confianza_error = y_pred_probs[indices_error, y_pred[indices_error]]
top_errores = indices_error[np.argsort(confianza_error)[::-1][:10]]

fig, ejes = plt.subplots(2, 5, figsize=(13, 5.5))
for idx, ax in zip(top_errores, ejes.flat):
    ax.imshow(X_test[idx], cmap="gray")
    ax.set_title(f"Real: {NOMBRES_CLASES[y_test[idx]]}\nPred: {NOMBRES_CLASES[y_pred[idx]]} "
                 f"({y_pred_probs[idx, y_pred[idx]]:.0%})", fontsize=8, color="crimson")
    ax.axis("off")
plt.suptitle("Los 10 errores con mayor confianza")
plt.tight_layout()
plt.show()

## 10. Probabilidades de una predicción individual

In [ ]:
idx = 21
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.8))
ejes[0].imshow(X_test[idx], cmap="gray")
ejes[0].set_title(f"Real: {NOMBRES_CLASES[y_test[idx]]}")
ejes[0].axis("off")
ejes[1].barh(NOMBRES_CLASES, y_pred_probs[idx], color="#2E86AB")
ejes[1].set_title("Probabilidad por clase")
plt.tight_layout()
plt.show()